In [100]:
import os
import json
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
import requests


In [116]:
requests.get("http://localhost:11434").content
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")




# to use openai api key from .env file, uncomment the following lines and make sure you have a .env file with OPENAI_API_KEY set
# load_dotenv(override=True)
# api_key = os.getenv("OPENAI_API_KEY")

# if not api_key:
#     print("Error: OPENAI_API_KEY environment variable is not set.")
# elif not api_key.startswith("sk-"):
#     print("Error: OPENAI_API_KEY does not appear to be a valid key.")
# elif api_key.strip() != api_key:
#     print("Error: OPENAI_API_KEY contains leading or trailing whitespace.") 
# else: 
#     print("OPENAI_API_KEY is set correctly.")
             

In [117]:
system_message = """
You are an helpful assistant for an Airline called FlightAI
Give short continuous answers not more than 1 line
Always be accurate, if you dont know the answer, say so"""

In [118]:
def chat(message, history):
    history = [{"role": h["role"], "content" : h["content"]} for h in history]
    messages = [{"role":"system", "content":system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(
        model = "qwen3:14b",
        messages= messages
    )
    return response.choices[0].message.content

gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7893
* To create a public link, set `share=True` in `launch()`.


In [119]:
ticket_prices = {"london": "$499", "paris": "$899", "berlin": "$299", "tokyo": "$899"}


def get_ticket_prices(destination_city):
    city = (destination_city or "").strip().lower()
    price = ticket_prices.get(city, "Unknown city")
    return f"The price of the ticket to {destination_city} is {price}"


In [120]:
print(get_ticket_prices("tokyo"))

The price of the ticket to tokyo is $899


In [121]:
price_function = {
    "name": "get_ticket_prices",
    "description": "Get the price of a return ticket to a destination city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to travel to"
            }
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}


In [122]:
tools =[{"type": "function", "function" : price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_prices',
   'description': 'Get the price of a return ticket to a destination city',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [126]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="qwen3:14b", messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model="qwen3:14b", messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [127]:
def handle_tool_calls(message):
    responses= []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_prices":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            price_details = get_ticket_prices(city)
            responses.append ({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses
    


In [128]:
gr.ChatInterface(chat).launch()


#llama 3.2 calling every response with tool_calls, so input destination city is not provided and hence fails
#gemma3 does not support tools
#gpt-oss also failing with complex questions
#Qwen3:14b works well with tool calling


* Running on local URL:  http://127.0.0.1:7895
* To create a public link, set `share=True` in `launch()`.


In [129]:
import sqlite3

In [132]:
DB = "prices.db"
with sqlite3.connect(DB) as conn:
    cursor= conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()


In [135]:
def get_ticket_prices(city):
    print(f"DATABASE tool called to get price of {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor= conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city= ?', (city.lower(),))
        result= cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No proce data available for the city"



In [136]:
get_ticket_prices("london")

DATABASE tool called to get price of london


'No proce data available for the city'

In [137]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [138]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [139]:
get_ticket_prices("Tokyo")

DATABASE tool called to get price of Tokyo


'Ticket price to Tokyo is $1420.0'

In [141]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7896
* To create a public link, set `share=True` in `launch()`.


DATABASE tool called to get price of london
DATABASE tool called to get price of Sydney
